In [ ]:
# Cell 1 — mount drive and install required pip packages
from google.colab import drive
# cleanly remount in case previous mount is flaky
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

# install packages you use; comment out ones you don't need
!pip install -q timm grad-cam lime torchinfo==1.7.0
print("✅ Drive mounted and packages installed")


Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 24.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 19.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
✅ Drive mounted and packages installed


In [ ]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
from efficientnet_pytorch import EfficientNet

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model
model = EfficientNet.from_pretrained("efficientnet-b0")
model._fc = torch.nn.Linear(model._fc.in_features, 2)
model.load_state_dict(torch.load("efficientnet_b0.pth", map_location=device))

model = model.to(device)
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels, _ in tqdm(val_loader):
        imgs = imgs.to(device)
        labels = labels.to(device)

        logits = model(imgs)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("DONE")


ModuleNotFoundError: No module named 'efficientnet_pytorch'

In [ ]:
ckpt = torch.load("model.pth", map_location="cpu")
print(ckpt["label_map"])


FileNotFoundError: [Errno 2] No such file or directory: 'model.pth'

In [ ]:
!ls -lh /content/drive/MyDrive/newssight/models/checkpoints
!ls -lh /content/drive/MyDrive/newssight/datasets/manifests | head


total 2.2G
-rw------- 1 root root 129M Oct  6 00:02 combined_resnet18_best.pth
-rw------- 1 root root 129M Oct  8 14:04 combined_resnet18_last_epoch.pth
-rw------- 1 root root 129M Sep 23 16:48 fakeddit_resnet18_best.pth
-rw------- 1 root root 129M Oct  5 20:33 periodic_epoch3_batch200.pth
-rw------- 1 root root 129M Oct  5 21:04 periodic_epoch3_batch400.pth
-rw------- 1 root root 129M Oct  5 21:35 periodic_epoch3_batch600.pth
-rw------- 1 root root 129M Oct  6 00:33 periodic_epoch4_batch200.pth
-rw------- 1 root root 129M Oct  6 01:04 periodic_epoch4_batch400.pth
-rw------- 1 root root 129M Oct  6 01:35 periodic_epoch4_batch600.pth
-rw------- 1 root root 129M Oct  6 02:58 periodic_epoch5_batch200.pth
-rw------- 1 root root 129M Oct  6 03:29 periodic_epoch5_batch400.pth
-rw------- 1 root root 129M Oct  6 03:59 periodic_epoch5_batch600.pth
-rw------- 1 root root 129M Oct  6 05:22 periodic_epoch6_batch200.pth
-rw------- 1 root root 129M Oct  8 10:31 periodic_epoch7_batch200.pth
-rw------

In [ ]:
import json, time, random, math
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T, torchvision.models as models
from sklearn.metrics import f1_score
from tqdm import tqdm

DRIVE_ROOT = Path("/content/drive/MyDrive/newssight")
CKPT_DIR = DRIVE_ROOT / "models" / "checkpoints"
BEST_CKPT = CKPT_DIR / "combined_resnet18_best.pth"
LAST_CKPT = CKPT_DIR / "combined_resnet18_last_epoch.pth"
TEMPERATURE_JSON = CKPT_DIR / "temperature.json"
VAL_MANIFEST = DRIVE_ROOT / "datasets" / "manifests" / "combined_val.csv"
GRADCAM_OUT = DRIVE_ROOT / "models" / "gradcam"
GRADCAM_OUT.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


In [ ]:
# --- Fixed ValDataset: handles 'fake'/'real' or numeric labels and keeps paths order ---
from pathlib import Path
from PIL import Image
import pandas as pd
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset

VAL_MANIFEST = Path("/content/drive/MyDrive/newssight/datasets/manifests/combined_val.csv")
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

# transforms (same as before)
val_tf = T.Compose([
    T.Resize(int(IMG_SIZE*1.02)),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class ValDataset(Dataset):
    def __init__(self, manifest_csv, transform):
        self.df = pd.read_csv(manifest_csv)
        if 'filepath' not in self.df.columns or 'label' not in self.df.columns:
            raise ValueError("Manifest must contain 'filepath' and 'label' columns.")
        # infer label mapping: if labels are strings (e.g., 'fake','real'), map them
        lab_values = list(self.df['label'].unique())
        # If labels are strings like 'fake'/'real', create mapping in sorted order for stability
        if any(isinstance(x, str) for x in lab_values):
            lab_sorted = sorted([str(x) for x in lab_values])
            self.label2idx = {lab_sorted[i]: i for i in range(len(lab_sorted))}
        else:
            # numeric labels (0/1) -> map sorted unique to 0..N-1 to be safe
            lab_sorted = sorted([int(x) for x in lab_values])
            self.label2idx = {lab_sorted[i]: i for i in range(len(lab_sorted))}
        # Build samples list (filepath, label_idx)
        samples = []
        for _, r in self.df.iterrows():
            fp = str(r['filepath'])
            raw_lab = r['label']
            # normalize raw_lab to str if mapping expects str
            if isinstance(list(self.label2idx.keys())[0], str):
                key = str(raw_lab)
            else:
                key = int(raw_lab)
            if key not in self.label2idx:
                # fallback: try converting types
                key = str(raw_lab) if isinstance(list(self.label2idx.keys())[0], str) else int(raw_lab)
            samples.append((fp, self.label2idx[key]))
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p, label = self.samples[idx]
        try:
            img = Image.open(p).convert('RGB')
        except Exception:
            # fallback black image
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0,0,0))
        if self.transform:
            img = self.transform(img)
        return img, label, p

# instantiate
val_ds = ValDataset(VAL_MANIFEST, transform=val_tf)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)

print("Val dataset size:", len(val_ds))
print("Label mapping:", val_ds.label2idx)
# print a few samples
for i in range(3):
    img, lab, path = val_ds[i]
    print(i, "label_idx:", lab, "path:", path)


Val dataset size: 28464
Label mapping: {'fake': 0, 'real': 1}
0 label_idx: 0 path: /content/drive/MyDrive/newssight/datasets/images/fakeddit/val/fake/train_fake_21377.jpg
1 label_idx: 0 path: /content/drive/MyDrive/newssight/datasets/images/fakeddit/val/fake/train_fake_100293.jpg
2 label_idx: 0 path: /content/drive/MyDrive/newssight/datasets/images/fakeddit/val/fake/train_fake_31286.jpg


In [ ]:
# Build model skeleton
num_classes = 2
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)
# Choose checkpoint to load
ckpt_path = BEST_CKPT if BEST_CKPT.exists() else LAST_CKPT if LAST_CKPT.exists() else None
if ckpt_path is None:
    raise FileNotFoundError("No checkpoint found on Drive. Re-run training or locate the checkpoint.")
ck = torch.load(str(ckpt_path), map_location=device)
# ck might wrap state dict as 'model_state_dict'
if 'model_state_dict' in ck:
    model.load_state_dict(ck['model_state_dict'])
else:
    model.load_state_dict(ck)
model.eval()
print("Loaded checkpoint:", ckpt_path)


Loaded checkpoint: /content/drive/MyDrive/newssight/models/checkpoints/combined_resnet18_best.pth


In [ ]:
all_logits = []
all_labels = []
paths_order = []  # keep filepaths in same order
with torch.no_grad():
    for imgs, labels, paths in tqdm(val_loader, desc="Collecting val logits"):
        imgs = imgs.to(device)
        outs = model(imgs)   # raw logits
        all_logits.append(outs.cpu())
        all_labels.append(torch.tensor(labels))
        paths_order.extend(paths)
all_logits = torch.cat(all_logits, dim=0)
all_labels = torch.cat(all_labels, dim=0)
print("Logits shape:", all_logits.shape, "Labels shape:", all_labels.shape)
# Optionally save to Drive to avoid recompute later:
torch.save({'logits': all_logits, 'labels': all_labels, 'paths': paths_order}, str(CKPT_DIR/'val_logits.pt'))
print("Saved val logits to", CKPT_DIR/'val_logits.pt')


  all_labels.append(torch.tensor(labels))

Logits shape: torch.Size([28464, 2]) Labels shape: torch.Size([28464])
Saved val logits to /content/drive/MyDrive/newssight/models/checkpoints/val_logits.pt


In [ ]:
# Cell 2 — imports + configuration
import os, time, math, random, shutil, json
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from tqdm import tqdm

# Torch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models
import torch.nn.functional as F

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# ---------- CONFIG ----------
DRIVE_ROOT = Path("/content/drive/MyDrive/newssight")
MANIFEST_DIR = DRIVE_ROOT / "datasets" / "manifests"
DRIVE_COMBINED_MANIFEST = MANIFEST_DIR / "combined_train.csv"
DRIVE_VAL_MANIFEST = MANIFEST_DIR / "combined_val.csv"

# local cache + manifests
LOCAL_CACHE_ROOT = Path("/tmp/newssight_cache")
LOCAL_TRAIN_CACHE = LOCAL_CACHE_ROOT / "train"
LOCAL_MANIFEST = LOCAL_CACHE_ROOT / "combined_train_local.csv"
LOCAL_VAL_MANIFEST = LOCAL_CACHE_ROOT / "combined_val_local.csv"

# copy settings
COPY_LIMIT = 60000          # <--- set 60000 as discussed
COPY_SKIP_EXISTING = True

# model / training hyperparams
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2             # set 0 while copying; increase after cache ready
NUM_EPOCHS = 8
LR = 1e-4
WEIGHT_DECAY = 1e-5
PATIENCE = 3
USE_AMP = True
SEED = 42

# checkpoints
CKPT_DIR = DRIVE_ROOT / "models" / "checkpoints"
LAST_CKPT = CKPT_DIR / "combined_resnet18_last_epoch.pth"
BEST_CKPT = CKPT_DIR / "combined_resnet18_best.pth"

# ensure dirs
def safe_makedirs(p: Path):
    p.mkdir(parents=True, exist_ok=True)
safe_makedirs(CKPT_DIR)
safe_makedirs(LOCAL_TRAIN_CACHE)

# seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = bool(USE_AMP and device.type == "cuda")
print(f"[CONFIG] device={device} use_amp={use_amp} COPY_LIMIT={COPY_LIMIT} BATCH_SIZE={BATCH_SIZE} NUM_WORKERS={NUM_WORKERS}")
# save config snapshot
safe_makedirs(LOCAL_CACHE_ROOT)
with open(LOCAL_CACHE_ROOT / "run_config.json", "w") as f:
    json.dump({
        "device": str(device), "use_amp": use_amp, "COPY_LIMIT": COPY_LIMIT,
        "BATCH_SIZE": BATCH_SIZE, "NUM_WORKERS": NUM_WORKERS, "NUM_EPOCHS": NUM_EPOCHS
    }, f, indent=2)


[CONFIG] device=cpu use_amp=False COPY_LIMIT=60000 BATCH_SIZE=32 NUM_WORKERS=2


In [ ]:
# Cell 3 — copy up to COPY_LIMIT files referenced in combined_train.csv into /tmp/newssight_cache/train
from pathlib import Path
import shutil, time
import pandas as pd
LOCAL_CACHE = LOCAL_TRAIN_CACHE
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

assert DRIVE_COMBINED_MANIFEST.exists(), f"Drive combined manifest missing: {DRIVE_COMBINED_MANIFEST}"
df_drive = pd.read_csv(DRIVE_COMBINED_MANIFEST)
files = df_drive['filepath'].tolist()
if COPY_LIMIT:
    files = files[:COPY_LIMIT]
print(f"[COPY] Preparing to copy {len(files)} files to {LOCAL_CACHE}")

copied, failed = 0, 0
start = time.time()
io_errors = []
for fp in tqdm(files, desc="Copying files"):
    src = Path(fp)
    try:
        ok = src.exists()
    except Exception as e:
        ok = False
        io_errors.append((str(src), str(e)))
    if not ok:
        failed += 1
        continue
    dst = LOCAL_CACHE / src.name
    try:
        if COPY_SKIP_EXISTING and dst.exists():
            copied += 1
            continue
        shutil.copy2(src, dst)
        copied += 1
    except Exception as e:
        failed += 1
        if len(io_errors) < 200:
            io_errors.append((str(src), str(e)))
        # small backoff occasionally
        if failed % 500 == 0:
            time.sleep(0.5)
        continue

elapsed = time.time() - start
print(f"[COPY] Done: copied={copied}, failed/skipped={failed}, elapsed={elapsed:.1f}s")
if io_errors:
    print("[COPY] sample IO errors:", io_errors[:5])

# build local manifest mapping to cached files only
rows = []
for _, r in df_drive.iterrows():
    name = Path(r['filepath']).name
    cand = LOCAL_CACHE / name
    if cand.exists():
        rows.append((str(cand), r['label']))
pd.DataFrame(rows, columns=['filepath','label']).to_csv(LOCAL_MANIFEST, index=False)
print("[COPY] Local manifest written:", LOCAL_MANIFEST, "rows:", len(rows))


[COPY] Preparing to copy 60000 files to /tmp/newssight_cache/train


Copying files: 100%|██████████| 60000/60000 [4:43:51<00:00,  3.52it/s]


[COPY] Done: copied=59997, failed/skipped=3, elapsed=17031.2s
[COPY] sample IO errors: [('/content/drive/MyDrive/newssight/datasets/raw/coco/train2017/000000341161.jpg', "[Errno 5] Input/output error: '/content/drive/MyDrive/newssight/datasets/raw/coco/train2017/000000341161.jpg'"), ('/content/drive/MyDrive/newssight/datasets/raw/coco/train2017/000000567085.jpg', "[Errno 5] Input/output error: '/content/drive/MyDrive/newssight/datasets/raw/coco/train2017/000000567085.jpg'"), ('/content/drive/MyDrive/newssight/datasets/raw/coco/train2017/000000399274.jpg', "[Errno 5] Input/output error: '/content/drive/MyDrive/newssight/datasets/raw/coco/train2017/000000399274.jpg'")]
[COPY] Local manifest written: /tmp/newssight_cache/combined_train_local.csv rows: 59997


In [ ]:
# Cell 4 — scan local cache for unreadable images, remove them, rewrite local manifest
from pathlib import Path
from PIL import Image
import pandas as pd

LOCAL_CACHE = LOCAL_TRAIN_CACHE
LOCAL_MANIFEST = LOCAL_MANIFEST
DRIVE_MANIFEST = DRIVE_COMBINED_MANIFEST

print("[CLEAN] Scanning cache:", LOCAL_CACHE)
bad = []
for p in LOCAL_CACHE.glob("*.jpg"):
    try:
        im = Image.open(p)
        im.verify()
    except Exception:
        bad.append(p)
print(f"[CLEAN] Found {len(bad)} corrupt images")
for p in bad:
    try:
        p.unlink()
    except Exception as e:
        print("Failed removing", p, e)
print("[CLEAN] Removed corrupt images if any")

# rebuild local manifest (match names)
if DRIVE_MANIFEST.exists():
    df_drive = pd.read_csv(DRIVE_MANIFEST)
    rows = []
    for _, r in df_drive.iterrows():
        name = Path(r['filepath']).name
        cand = LOCAL_CACHE / name
        if cand.exists():
            rows.append((str(cand), r['label']))
    pd.DataFrame(rows, columns=['filepath','label']).to_csv(LOCAL_MANIFEST, index=False)
    print("[CLEAN] Rewrote local manifest:", LOCAL_MANIFEST, "rows:", len(rows))
else:
    print("[CLEAN] Drive manifest missing; cannot rebuild local manifest.")


[CLEAN] Scanning cache: /tmp/newssight_cache/train
[CLEAN] Found 2 corrupt images
[CLEAN] Removed corrupt images if any
[CLEAN] Rewrote local manifest: /tmp/newssight_cache/combined_train_local.csv rows: 59995


In [ ]:
# Cell 5 — build CSVDatasetSafe, transforms, and dataloaders
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
from PIL import Image
import numpy as np
import pandas as pd
import random

LOCAL_MANIFEST = LOCAL_MANIFEST
LOCAL_VAL_MANIFEST = LOCAL_VAL_MANIFEST

IMG_SIZE = IMG_SIZE
BATCH_SIZE = BATCH_SIZE

# transforms
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.ColorJitter(0.2,0.2,0.2,0.02)], p=0.5),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_tf = T.Compose([
    T.Resize(int(IMG_SIZE*1.02)),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class CSVDatasetSafe(Dataset):
    def __init__(self, manifest_csv, transform=None, max_retries=3):
        self.df = pd.read_csv(manifest_csv)
        unique_labels = sorted(self.df['label'].unique())
        self.label2idx = {lab: i for i, lab in enumerate(unique_labels)}
        self.idx2label = {i:lab for lab,i in self.label2idx.items()}
        self.samples = [(str(r['filepath']), self.label2idx[r['label']]) for _, r in self.df.iterrows()]
        self.transform = transform
        self.max_retries = max_retries

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        for attempt in range(self.max_retries):
            path, label = self.samples[idx]
            try:
                img = Image.open(path).convert('RGB')
                if self.transform:
                    img = self.transform(img)
                return img, int(label)
            except Exception as e:
                if attempt == 0:
                    print(f"[DATA] Warning: failed to load {path}: {e}")
                idx = random.randint(0, len(self.samples)-1)
                continue
        raise IndexError("Cannot load sample after retries")

# create datasets
assert LOCAL_MANIFEST.exists(), f"Local manifest missing: {LOCAL_MANIFEST}"
train_ds = CSVDatasetSafe(LOCAL_MANIFEST, transform=train_tf)
# val prefer local val if exists, else drive val
if LOCAL_VAL_MANIFEST.exists():
    val_ds = CSVDatasetSafe(LOCAL_VAL_MANIFEST, transform=val_tf)
else:
    assert DRIVE_VAL_MANIFEST.exists(), "No validation manifest found"
    val_ds = CSVDatasetSafe(str(DRIVE_VAL_MANIFEST), transform=val_tf)

# Weighted sampler
labels = [lab for _, lab in train_ds.samples]
labels_arr = np.array(labels, dtype=np.int64)
max_label = labels_arr.max() if labels_arr.size else 0
class_counts = np.bincount(labels_arr, minlength=max_label+1)
class_weights = 1.0 / (class_counts.astype(np.float64) + 1e-8)
samples_weights = [class_weights[int(l)] for l in labels_arr]
sampler = WeightedRandomSampler(weights=samples_weights, num_samples=len(samples_weights), replacement=True)

# dataloaders
num_workers = NUM_WORKERS if device.type=='cuda' else 0
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=num_workers, pin_memory=(device.type=='cuda'))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=(device.type=='cuda'))

print("[DATA] classes:", train_ds.label2idx)
print("[DATA] train samples:", len(train_ds), "val samples:", len(val_ds), "num_workers:", num_workers)


AssertionError: Local manifest missing: /tmp/newssight_cache/combined_train_local.csv

In [ ]:
# Cell 6 — model init, optimizer, scheduler, resume support
import math, time
from pathlib import Path

CKPT_DIR = CKPT_DIR
LAST_CKPT = LAST_CKPT
BEST_CKPT = BEST_CKPT

device = device
use_amp = use_amp

# build model
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, len(train_ds.label2idx))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# resume if last checkpoint exists
start_epoch = 1
best_val_f1 = 0.0
if LAST_CKPT.exists():
    try:
        ck = torch.load(LAST_CKPT, map_location=device)
        model.load_state_dict(ck['model_state_dict'])
        optimizer.load_state_dict(ck.get('optimizer_state_dict', optimizer.state_dict()))
        start_epoch = ck.get('epoch', 1) + 1
        best_val_f1 = ck.get('best_val_f1', 0.0)
        print(f"[CKPT] Resumed from {LAST_CKPT} epoch {start_epoch-1} best_val_f1={best_val_f1}")
    except Exception as e:
        print("[CKPT] Warning: failed to fully restore checkpoint:", e)

# evaluation helper (returns loss, acc, prec, rec, f1, cm)
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device); labels = labels.to(device)
            if use_amp:
                with torch.cuda.amp.autocast():
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1).detach().cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())
    if len(all_labels)==0:
        return 0.0, 0.0, 0.0, 0.0, 0.0, None
    val_loss = total_loss / len(loader.dataset)
    acc = (np.array(all_preds) == np.array(all_labels)).mean()
    if len(train_ds.label2idx) == 2:
        pos_label = train_ds.label2idx.get('fake', 1)
        prec = precision_score(all_labels, all_preds, average='binary', zero_division=0, pos_label=pos_label)
        rec = recall_score(all_labels, all_preds, average='binary', zero_division=0, pos_label=pos_label)
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0, pos_label=pos_label)
    else:
        prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
        rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
        f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    return val_loss, acc, prec, rec, f1, cm

print("[MODEL] Ready. Starting from epoch", start_epoch)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 136MB/s]
/tmp/ipython-input-2121329763.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


[CKPT] Resumed from /content/drive/MyDrive/newssight/models/checkpoints/combined_resnet18_last_epoch.pth epoch 8 best_val_f1=0.7249438041888179
[MODEL] Ready. Starting from epoch 9


In [ ]:
# Cell 7 — training loop
import time, math
from tqdm import tqdm

NUM_EPOCHS = NUM_EPOCHS
PATIENCE = PATIENCE
epochs_no_improve = 0

print("[TRAIN] Starting training on device:", device)
try:
    for epoch in range(start_epoch, NUM_EPOCHS+1):
        t0 = time.time()
        model.train()
        running_loss = 0.0
        all_preds, all_labels = [], []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}", leave=True)
        for imgs, labels in pbar:
            imgs = imgs.to(device); labels = labels.to(device)
            optimizer.zero_grad()
            if use_amp:
                with torch.cuda.amp.autocast():
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1).detach().cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())
            pbar.set_postfix(loss=running_loss / (len(all_preds)+1e-9))

        # training metrics
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = (np.array(all_preds) == np.array(all_labels)).mean()
        if len(train_ds.label2idx) == 2:
            pos_label = train_ds.label2idx.get('fake', 1)
            train_f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0, pos_label=pos_label)
        else:
            train_f1 = f1_score(all_preds, all_labels, average='macro', zero_division=0)

        # validation
        val_loss, val_acc, val_prec, val_rec, val_f1, cm = evaluate(model, val_loader)
        if not math.isnan(val_f1):
            scheduler.step(val_f1)

        elapsed = time.time() - t0
        print(f"[EPOCH] {epoch}/{NUM_EPOCHS} time {elapsed:.0f}s")
        print(f"  train_loss {train_loss:.4f} train_acc {train_acc:.4f} train_f1 {train_f1:.4f}")
        print(f"  val_loss   {val_loss:.4f} val_acc {val_acc:.4f} val_f1 {val_f1:.4f} val_prec {val_prec:.4f} val_rec {val_rec:.4f}")
        print("  Confusion matrix:\n", cm)

        # save last epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_f1': best_val_f1
        }, LAST_CKPT)

        # save best
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_f1': val_f1,
                'label_map': train_ds.label2idx
            }, BEST_CKPT)
            print("  ✅ Saved BEST model to", BEST_CKPT)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f"  No improvement epochs: {epochs_no_improve}/{PATIENCE}")

        if epochs_no_improve >= PATIENCE:
            print("[TRAIN] Early stopping triggered.")
            break

except Exception as ex:
    print("[TRAIN] Training crashed with exception:", ex)
    # try to save crash checkpoint
    try:
        crash_path = CKPT_DIR / f"crash_ckpt_epoch{epoch}.pth"
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_f1': best_val_f1,
            'crash_exc': str(ex)
        }, crash_path)
        print("[TRAIN] Saved crash checkpoint to", crash_path)
    except Exception as e:
        print("[TRAIN] Failed saving crash checkpoint:", e)
    raise

print("[TRAIN] Finished. Best val F1:", best_val_f1)


[TRAIN] Starting training on device: cpu
[TRAIN] Finished. Best val F1: 0.7249438041888179


In [ ]:
# === Cell 8: Robustly collect val logits (supports (img,label) or (img,label,path)), then do temperature scaling ===
import torch, torch.nn as nn, torch.optim as optim, json
import torch.nn.functional as F
from tqdm import tqdm
from pathlib import Path

# Paths (adjust if your variables differ)
CKPT_DIR = Path("/content/drive/MyDrive/newssight/models/checkpoints")
LOCAL_CACHE = Path("/tmp/newssight_cache")
LOGITS_PTH = LOCAL_CACHE / "val_logits.pth"
TEMP_JSON = CKPT_DIR / "temperature.json"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# 1) load model from best or last checkpoint
ckpt_path = None
if 'BEST_CKPT' in globals() and BEST_CKPT.exists():
    ckpt_path = BEST_CKPT
elif 'LAST_CKPT' in globals() and LAST_CKPT.exists():
    ckpt_path = LAST_CKPT
else:
    raise RuntimeError("No checkpoint found (BEST_CKPT or LAST_CKPT). Load/resume training first.")

print("Loading checkpoint:", ckpt_path)
ck = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ck['model_state_dict'])
model.to(device).eval()

# 2) collect logits/labels/paths if not already saved
if LOGITS_PTH.exists():
    print("Found existing logits file:", LOGITS_PTH, "- loading.")
    data = torch.load(LOGITS_PTH)
    logits = data['logits']
    labels = data['labels']
    paths = data.get('paths', [None] * len(labels))
    print("Loaded logits:", logits.shape, "labels:", labels.shape, "paths:", len(paths))
else:
    print("No saved logits found. Running val pass to collect logits (this may take a while)...")
    logits_list, labels_list, paths_list = [], [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Collecting val logits"):
            # accept (imgs,labels) OR (imgs,labels,paths)
            if isinstance(batch, (list,tuple)) and len(batch) == 3:
                imgs, labels_batch, batch_paths = batch
            elif isinstance(batch, (list,tuple)) and len(batch) == 2:
                imgs, labels_batch = batch
                batch_paths = [None] * imgs.size(0)
            else:
                # try to pick first two items
                imgs = batch[0]
                labels_batch = batch[1]
                batch_paths = [None] * imgs.size(0)

            imgs = imgs.to(device)
            out = model(imgs)                # logits
            logits_list.append(out.detach().cpu())
            labels_list.append(labels_batch.detach().cpu())
            # normalize paths to strings or None
            if isinstance(batch_paths, (list,tuple)):
                batch_paths = [str(p) if p is not None else None for p in batch_paths]
            else:
                # tensor? fallback
                batch_paths = [None] * imgs.size(0)
            paths_list.extend(batch_paths)

    logits = torch.cat(logits_list, dim=0)
    labels = torch.cat(labels_list, dim=0).long()
    paths = paths_list
    safe_dir = LOGITS_PTH.parent
    safe_dir.mkdir(parents=True, exist_ok=True)
    torch.save({'logits': logits, 'labels': labels, 'paths': paths}, LOGITS_PTH)
    print("Saved logits to:", LOGITS_PTH, "shapes:", logits.shape, labels.shape)

# 3) temperature scaling using LBFGS on CPU (single scalar)
print("Running temperature scaling (LBFGS) on CPU...")
logits_cpu = logits.clone().cpu()
labels_cpu = labels.clone().cpu()

temperature = nn.Parameter(torch.ones(1) * 1.0, requires_grad=True)
nll = nn.CrossEntropyLoss()
optimizer_t = optim.LBFGS([temperature], lr=0.01, max_iter=200, history_size=10)

def nll_loss():
    T = temperature.clamp(min=1e-6)
    scaled = logits_cpu / T
    return nll(scaled, labels_cpu)

def closure():
    optimizer_t.zero_grad()
    loss = nll_loss()
    loss.backward()
    return loss

try:
    optimizer_t.step(closure)
except Exception as e:
    print("LBFGS failed:", e)

T_final = float(temperature.detach().cpu().clamp(min=1e-6).item())
print("Temperature found:", T_final)

# save
CKPT_DIR.mkdir(parents=True, exist_ok=True)
with open(TEMP_JSON, "w") as f:
    json.dump({"temperature": T_final}, f)
print("Saved temperature to:", TEMP_JSON)


Device: cpu
Loading checkpoint: /content/drive/MyDrive/newssight/models/checkpoints/combined_resnet18_best.pth
No saved logits found. Running val pass to collect logits (this may take a while)...


/usr/local/lib/python3.12/dist-packages/torch/optim/lbfgs.py:457: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  loss = float(closure())


Saved logits to: /tmp/newssight_cache/val_logits.pth shapes: torch.Size([28464, 2]) torch.Size([28464])
Running temperature scaling (LBFGS) on CPU...
Temperature found: 1.9617403745651245
Saved temperature to: /content/drive/MyDrive/newssight/models/checkpoints/temperature.json


In [ ]:
# Cell 9 — Generate Grad-CAM overlays + thumbnails + JSON metadata (defensive cleanup)
# Paste into Colab and run (requires pytorch-grad-cam installed in Cell 1)
import json, gc, os
from pathlib import Path
from tqdm import tqdm
import numpy as np
import cv2
from PIL import Image
import torch
import torch.nn.functional as F

# pytorch-grad-cam imports
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# --- CONFIG / PATHS ---
DRIVE_ROOT = Path("/content/drive/MyDrive/newssight")
CKPT_DIR = DRIVE_ROOT / "models" / "checkpoints"
BEST_CKPT = CKPT_DIR / "combined_resnet18_best.pth"
LAST_CKPT = CKPT_DIR / "combined_resnet18_last_epoch.pth"
OUT_DIR = DRIVE_ROOT / "models" / "gradcam"
THUMB_SIZE = (256, 256)
N_CAM = 200   # how many CAMs to generate (misclassified prioritized)
safe_makedirs(OUT_DIR)

# ensure model & val_loader exist in the notebook environment (re-run setup cells 1-7 if needed)
assert 'model' in globals(), "Model not loaded - re-run setup cells (model creation + ckpt load)."
assert 'val_loader' in globals() or (Path("/tmp/newssight_cache/combined_val_local.csv").exists() or (DRIVE_ROOT/"datasets"/"manifests"/"combined_val.csv").exists()), \
    "Val loader or manifest missing - re-run dataset cells."

# load checkpoint weights into model (prefer BEST then LAST)
ckpt_path = BEST_CKPT if BEST_CKPT.exists() else LAST_CKPT if LAST_CKPT.exists() else None
assert ckpt_path is not None, "No checkpoint found; run training first."
ck = torch.load(ckpt_path, map_location='cpu')
model.load_state_dict(ck['model_state_dict'])
model = model.to('cuda' if torch.cuda.is_available() else 'cpu').eval()

# load temperature if present
T_val = 1.0
temp_path = CKPT_DIR / "temperature.json"
if temp_path.exists():
    try:
        T_val = float(json.load(open(temp_path))["temperature"])
    except Exception:
        T_val = 1.0
print("[GRADCAM] using temperature:", T_val)

# build val dataframe (prefer Drive manifest, fall back to local cached manifest)
VAL_MANIFEST_ON_DRIVE = DRIVE_ROOT / "datasets" / "manifests" / "combined_val.csv"
LOCAL_VAL_MANIFEST = Path("/tmp/newssight_cache/combined_val_local.csv")
if VAL_MANIFEST_ON_DRIVE.exists():
    val_df = pd.read_csv(VAL_MANIFEST_ON_DRIVE)
elif LOCAL_VAL_MANIFEST.exists():
    val_df = pd.read_csv(LOCAL_VAL_MANIFEST)
else:
    raise FileNotFoundError("No val manifest found (drive or local).")

# safe function to get image path (prefer local cache)
def resolve_path(fp: str):
    fname = Path(fp).name
    local_cache = Path("/tmp/newssight_cache") / "train"  # where you copied train images
    local_val_cache = Path("/tmp/newssight_cache") / "val"
    cand1 = local_val_cache / fname
    cand2 = local_cache / fname
    cand3 = Path(fp)
    for c in (cand1, cand2, cand3):
        if c.exists():
            return c
    return cand3  # return original even if missing (will be handled later)

# Compute predictions on val set to select misclassified images
print("Computing predictions on val set to prioritize misclassified images...")
paths_all, gts_all, preds_all, probs_all = [], [], [], []
# We'll iterate val_df directly to avoid differing DataLoader shapes
batch_tf = val_tf  # reuse your validation transform from previous cells
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="Val pass (fast)"):
    fp = str(row['filepath'])
    img_path = resolve_path(fp)
    try:
        bgr = cv2.imread(str(img_path))
        if bgr is None:
            # unreadable -> skip
            continue
        rgb = bgr[:, :, ::-1].astype(np.float32) / 255.0
        pil = Image.fromarray((rgb * 255).astype(np.uint8))
        inp = batch_tf(pil).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(inp)
            if T_val != 1.0:
                soft = F.softmax(logits / T_val, dim=1)
            else:
                soft = F.softmax(logits, dim=1)
            prob = float(soft.cpu().numpy().squeeze().max())
            pred = int(logits.argmax(dim=1).cpu().item())
        paths_all.append(str(img_path))
        gts_all.append(int(row['label'] if isinstance(row['label'], (int, np.integer)) else (1 if row['label']=="real" else 0)))
        preds_all.append(pred)
        probs_all.append(prob)
    except Exception:
        continue

if len(paths_all) == 0:
    raise RuntimeError("No readable validation images found - check manifest/cache.")

mis_idx = [i for i,(g,p) in enumerate(zip(gts_all,preds_all)) if g!=p]
other_idx = [i for i in range(len(paths_all)) if i not in mis_idx]
selected_idx = (mis_idx + other_idx)[:N_CAM]
print(f"Total val samples: {len(paths_all)}; misclassified: {len(mis_idx)}; will compute CAMs for {len(selected_idx)} images.")

# prepare output metadata container
meta = []
count = 0

# For each selected index, compute Grad-CAM and save images + thumbnails + metadata
for i in tqdm(selected_idx, desc="Generating GradCAM overlays"):
    img_path = Path(paths_all[i])
    try:
        orig_bgr = cv2.imread(str(img_path))
        if orig_bgr is None:
            # skip unreadable
            continue
        rgb = orig_bgr[:, :, ::-1].astype(np.float32) / 255.0

        # prepare input tensor
        pil = Image.fromarray((rgb * 255).astype(np.uint8))
        inp_tensor = batch_tf(pil).unsqueeze(0).to(device)

        # pick target layer (ResNet last conv block)
        try:
            target_layer = model.layer4[-1]
        except Exception:
            # fallback: use last child module if structure differs
            target_layer = list(model.named_children())[-2][1]

        # instantiate GradCAM (no deprecated kwargs)
        cam = GradCAM(model=model, target_layers=[target_layer])

        # forward and get CAM for predicted class
        with torch.no_grad():
            logits = model(inp_tensor)
        pred_class = int(logits.argmax(dim=1).cpu().item())
        targets = [ClassifierOutputTarget(pred_class)]

        try:
            grayscale_cam = cam(input_tensor=inp_tensor, targets=targets)[0]
        except Exception as e:
            # if cam fails for this image, clean and skip
            print("CAM computation failed for", img_path, e)
            try:
                if hasattr(cam, "activations_and_grads"):
                    try:
                        cam.activations_and_grads.release()
                    except Exception:
                        pass
            except Exception:
                pass
            try:
                del cam
            except Exception:
                pass
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            continue

        visualization = show_cam_on_image(rgb, grayscale_cam, use_rgb=True)  # returns HxWx3 uint8

        # file names
        base_name = f"gradcam_{count}_{img_path.stem}.jpg"
        thumb_name = f"gradcam_{count}_{img_path.stem}_thumb.jpg"
        out_full = OUT_DIR / base_name
        out_thumb = OUT_DIR / thumb_name

        # save full overlay (BGR expected by cv2)
        cv2.imwrite(str(out_full), visualization[:, :, ::-1])

        # save thumbnail (PIL)
        thumb = Image.fromarray(visualization).convert("RGB").resize(THUMB_SIZE, Image.LANCZOS)
        thumb.save(out_thumb, format="JPEG", quality=90, optimize=True)

        # gather metadata (store original path, predicted label, ground truth if present, confidence, files)
        gt = gts_all[i] if i < len(gts_all) else None
        pred = preds_all[i] if i < len(preds_all) else int(pred_class)
        conf = float(probs_all[i]) if i < len(probs_all) else float(F.softmax(logits / T_val, dim=1).cpu().numpy().squeeze().max())

        rec = {
            "orig_filepath": str(img_path),
            "out_overlay": str(out_full),
            "out_thumbnail": str(out_thumb),
            "pred_label": int(pred),
            "gt_label": int(gt) if gt is not None else None,
            "confidence": float(conf),
            "index": int(i)
        }
        meta.append(rec)
        count += 1

        # defensive cleanup for cam instance
        try:
            if hasattr(cam, "activations_and_grads"):
                try:
                    cam.activations_and_grads.release()
                except Exception:
                    pass
        except Exception:
            pass
        try:
            del cam
        except Exception:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        print("Unexpected error processing", img_path, e)
        continue

# save metadata list (full manifest)
meta_path = OUT_DIR / "manifest_gradcam.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[GRADCAM] Generated {len(meta)} overlays → {OUT_DIR}")
print("[GRADCAM] Manifest written to", meta_path)


[GRADCAM] using temperature: 1.9617403745651245
Computing predictions on val set to prioritize misclassified images...


Val pass (fast): 100%|██████████| 28464/28464 [35:55<00:00, 13.21it/s]


Total val samples: 28464; misclassified: 7709; will compute CAMs for 200 images.


Generating GradCAM overlays: 100%|██████████| 200/200 [01:36<00:00,  2.07it/s]

[GRADCAM] Generated 200 overlays → /content/drive/MyDrive/newssight/models/gradcam
[GRADCAM] Manifest written to /content/drive/MyDrive/newssight/models/gradcam/manifest_gradcam.json
